# Study 948 — Capture Ratio — the teardown

The capture spread as an estimand: a joint block-bootstrap CI, the Henriksson-Merton convexity twin with a HAC *t* judged against a Bonferroni bar, CAPM alphas, the excess-of-cash Sharpe race, an era cut with rank persistence, the assumption sweeps, the beta-neutral traded arm with cost × borrow, and the live synthetic control.

**Design.** Monthly simple **total** returns (`auto_adjust=True`), months partitioned by the *benchmark's* sign, cash leg BIL, per-fund windows from each fund's own inception, 2007-06-30 → 2026-06-30. **One execution lag, and only in the traded arm**: the 36-month rolling beta is estimated through month *t* and held over *t*+1. Capture ratios are a contemporaneous attribution — nothing is traded on them, so no lag applies. Costs are one-way bps × NAV; the short index leg pays borrow; both are swept.

Every real number is frozen from `docs/results.md` (fingerprint `01c183fa5741`).

In [1]:
R = {'start': '2007-06-30', 'end': '2026-06-30', 'n_obs': 327, 'fp': '01c183fa5741', 'n_funds': 14, 'bonferroni': 2.91, 'spread_median': -0.013, 'ci_covers_zero': 14, 'max_abs_t_alpha': 1.56, 'n_beating_bench': 1, 'rho': -0.117, 'rho_p': 0.756, 'rho_n': 9, 'sign_agreement': 0.44, 'matched_n_sig': 0, 'matched_min_p': 0.127, 'matched_median_excess': -0.035, 'null_spread_mean': 0.111, 'null_frac_pos': 0.82}

## 0. Data hygiene first — one unadjusted corporate action

`data.split_artifacts` sweeps all eighteen daily series for one-day price ratios that land on a real share-count factor (3:2, 2:1, 3:1, 4:1, 5:1, 10:1 and their reverse twins, 3% tolerance, ±35% jump threshold). The whitelist is deliberately short: a dense list of every *n/m* would tile the number line and smooth a genuine crash away. It fires **exactly once**.

In [2]:
R_SPLIT = {'ticker': 'NUSI', 'date': '2025-02-18', 'ratio': 2.0, 'raw': {'down': -0.309, 'spread': 0.939, 'alpha': 185.6, 't_alpha': 1.25, 'traded': 331.5}, 'fixed': {'down': 0.466, 'spread': 0.164, 'alpha': 33.6, 't_alpha': 1.43, 'traded': 53.0}, 'ci0_raw': 13, 'ci0_fixed': 14}
print(f"{R_SPLIT['ticker']} {R_SPLIT['date']}: one-day ratio "
      f"{R_SPLIT['ratio']:.6f} -> read as a 1-for-2 reverse split, back-adjusted")
print()
print(f"{'':10}{'down-cap':>10}{'spread':>9}{'alpha bps':>11}{'t':>7}{'traded bps':>12}")
for tag in ('raw', 'fixed'):
    d = R_SPLIT[tag]
    print(f"{tag:10}{d['down']:+10.3f}{d['spread']:+9.3f}"
          f"{d['alpha']:+11.1f}{d['t_alpha']:+7.2f}{d['traded']:+12.1f}")
print()
print(f"panel CI-covers-zero count: {R_SPLIT['ci0_raw']} raw -> "
      f"{R_SPLIT['ci0_fixed']} repaired (of 14)")


NUSI 2025-02-18: one-day ratio 2.000000 -> read as a 1-for-2 reverse split, back-adjusted

            down-cap   spread  alpha bps      t  traded bps
raw           -0.309   +0.939     +185.6  +1.25      +331.5
fixed         +0.466   +0.164      +33.6  +1.43       +53.0

panel CI-covers-zero count: 13 raw -> 14 repaired (of 14)


`auto_adjust=True` is supposed to handle this and did not. Left in, the fake +100% month made NUSI the panel's only fund with a CI excluding zero — i.e. it would have been the study's one 'result'. Everything below runs on the repaired tape; `data.load_prices(repair_splits=False)` reproduces the contaminated one.

## 1. The scorecard

`spread` = up-capture − down-capture (arithmetic, ratio of means). CI = 95% circular block bootstrap, 2,000 draws, 3-month blocks, fund and benchmark resampled **jointly**. `convexity t` = HAC (Newey-West, 6 lags) *t* on `b_up − b_dn` from the piecewise fit on excess-of-cash returns.

In [3]:
R_SCORE = {'QYLD': ('QQQ', 150, 0.497, 0.559, -0.062, -0.21, 0.11, 0.23, -0.35, -2.62), 'JEPQ': ('QQQ', 49, 0.712, 0.689, 0.023, -0.13, 0.24, 0.64, -0.44, -5.58), 'NUSI': ('QQQ', 78, 0.63, 0.466, 0.164, -0.03, 0.39, 0.95, -0.2, -1.65), 'XYLD': ('SPY', 156, 0.641, 0.711, -0.07, -0.22, 0.1, 0.2, -0.35, -2.14), 'JEPI': ('SPY', 73, 0.592, 0.571, 0.021, -0.16, 0.2, 0.6, -0.14, -0.72), 'PBP': ('SPY', 222, 0.554, 0.614, -0.06, -0.19, 0.09, 0.21, -0.37, -3.69), 'DIVO': ('SPY', 114, 0.772, 0.729, 0.042, -0.1, 0.19, 0.73, 0.01, 0.07), 'SPYI': ('SPY', 46, 0.745, 0.72, 0.026, -0.05, 0.16, 0.71, -0.36, -5.49), 'SCHD': ('SPY', 176, 0.86, 0.839, 0.02, -0.18, 0.24, 0.56, -0.09, -0.69), 'VYM': ('SPY', 229, 0.865, 0.873, -0.008, -0.14, 0.13, 0.46, -0.1, -1.23), 'DVY': ('SPY', 229, 0.8, 0.819, -0.018, -0.24, 0.21, 0.46, -0.21, -1.39), 'SPHD': ('SPY', 164, 0.713, 0.754, -0.041, -0.34, 0.27, 0.39, -0.28, -1.16), 'NOBL': ('SPY', 152, 0.822, 0.894, -0.072, -0.28, 0.13, 0.24, -0.15, -0.85), 'RYLD': ('IWM', 86, 0.565, 0.593, -0.028, -0.21, 0.2, 0.43, -0.53, -3.15)}
print(f"{'fund':6}{'bench':6}{'n':>5}{'up':>8}{'down':>8}{'spread':>9}{'95% CI':>18}{'conv t':>9}")
for f, (b, n, up, dn, sp, lo, hi, p, cv, t) in R_SCORE.items():
    ci = f'[{lo:+.2f}, {hi:+.2f}]'
    print(f"{f:6}{b:6}{n:5d}{up:8.3f}{dn:8.3f}{sp:+9.3f}{ci:>18}{t:+9.2f}")


fund  bench     n      up    down   spread            95% CI   conv t
QYLD  QQQ     150   0.497   0.559   -0.062    [-0.21, +0.11]    -2.62
JEPQ  QQQ      49   0.712   0.689   +0.023    [-0.13, +0.24]    -5.58
NUSI  QQQ      78   0.630   0.466   +0.164    [-0.03, +0.39]    -1.65
XYLD  SPY     156   0.641   0.711   -0.070    [-0.22, +0.10]    -2.14
JEPI  SPY      73   0.592   0.571   +0.021    [-0.16, +0.20]    -0.72
PBP   SPY     222   0.554   0.614   -0.060    [-0.19, +0.09]    -3.69
DIVO  SPY     114   0.772   0.729   +0.042    [-0.10, +0.19]    +0.07
SPYI  SPY      46   0.745   0.720   +0.026    [-0.05, +0.16]    -5.49
SCHD  SPY     176   0.860   0.839   +0.020    [-0.18, +0.24]    -0.69
VYM   SPY     229   0.865   0.873   -0.008    [-0.14, +0.13]    -1.23
DVY   SPY     229   0.800   0.819   -0.018    [-0.24, +0.21]    -1.39
SPHD  SPY     164   0.713   0.754   -0.041    [-0.34, +0.27]    -1.16
NOBL  SPY     152   0.822   0.894   -0.072    [-0.28, +0.13]    -0.85
RYLD  IWM      86   

In [4]:
N_POS, N_BONF, N_NEG = (0, 0, 6)
CONCAVE = ['QYLD', 'JEPQ', 'XYLD', 'PBP', 'SPYI', 'RYLD']
print(f"Bonferroni bar for {R['n_funds']} simultaneous tests: "
      f"|t| >= {R['bonferroni']:.2f}")
print(f"funds with convexity t >= +2 (nominal) : {N_POS}")
print(f"funds clearing the Bonferroni bar      : {N_BONF}")
print(f"funds with convexity t <= -2 (CONCAVE) : {N_NEG}  -> {CONCAVE}")
print(f"bootstrap CI covers zero for {R['ci_covers_zero']}/{R['n_funds']} funds")
print(f"cross-sectional spread: median {R['spread_median']:+.3f}")


Bonferroni bar for 14 simultaneous tests: |t| >= 2.91
funds with convexity t >= +2 (nominal) : 0
funds clearing the Bonferroni bar      : 0
funds with convexity t <= -2 (CONCAVE) : 6  -> ['QYLD', 'JEPQ', 'XYLD', 'PBP', 'SPYI', 'RYLD']
bootstrap CI covers zero for 14/14 funds
cross-sectional spread: median -0.013


> 💡 **In plain words** — nothing in the table is convex. The only funds whose payoff shape is statistically distinguishable from a straight line are bent the *wrong* way, and they are exactly the funds that write options.

## 2. The estimator's null bias — why the ratio cannot be read at face value

The capture spread is a difference of two ratios of sample means whose down-leg denominator is a small negative number. Ratio estimators of that shape are biased in finite samples. We measure it directly on zero-truth synthetic panels (**synthetic by construction** — this is a property of the estimator, not of any fund).

In [5]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from capture_ratio import data, strategy as st
nb = st.null_spread_distribution(
    data.synthetic_panel(signal_strength=0.0, seed=948 + s) for s in range(8))
print(f"{nb['n_obs']} fund-panels, planted true capture spread = 0.000 for every one")
print(f"arithmetic spread : mean {nb['spread_mean']:+.3f}  median {nb['spread_median']:+.3f}"
      f"  sd {nb['spread_sd']:.3f}  positive {nb['spread_frac_positive']:.0%}")
print(f"HM convexity      : mean {nb['convexity_mean']:+.3f}  "
      f"median {nb['convexity_median']:+.3f}  sd {nb['convexity_sd']:.3f}  (unbiased)")


56 fund-panels, planted true capture spread = 0.000 for every one
arithmetic spread : mean +0.107  median +0.058  sd 0.119  positive 84%
HM convexity      : mean +0.003  median -0.000  sd 0.067  (unbiased)


On the published run this is **mean +0.111, positive 82% of the time** for the ratio versus **-0.003** for the regression twin.

**This number must not be subtracted from the real panel.** It is an *unconditional* bias — it averages over random benchmark paths. A real capture spread is computed on **one** realised benchmark path, and conditioning on that path removes most of it. Quoting +0.111 against the real median would be mixing tapes and would overstate the case roughly threefold. The real-tape version is section 2b.

## 2b. The fund-matched null — the bias measured on each fund's own sample

`strategy.matched_null_spread` rebuilds each fund 1,000 times as a **zero-convexity twin**: the fund's own beta and residual vol (circular block bootstrap of the OLS residuals) on the **real** benchmark and cash paths, re-scored with the same estimator. Two variants:

* **null(β)** — zero alpha too, i.e. a plain scaled index position. This is the test, and it has power: on the planted synthetic panel the genuinely convex fund clears it at *p* = 0.000 and the concave ones sit at the far tail.
* **null(α,β)** — keeps the fund's own fitted alpha. A **decomposition, not a test**.

**Stated because it cuts against us:** null(β) also fires on a perfectly *linear* fund carrying a positive alpha, since the capture spread of a linear fund is `a·(1/mean_up − 1/mean_down)` and `mean_down < 0`, so the two terms add. A small *p* there means 'beats a scaled index', **not** 'is convex'. Convexity as such is the HM column in section 1, whose free intercept separates the two.

In [6]:
R_MATCH = {'QYLD': (-0.062, 0.031, -0.093, 0.886, -0.058, -0.004), 'JEPQ': (0.023, 0.052, -0.029, 0.644, 0.069, -0.046), 'NUSI': (0.164, 0.047, 0.117, 0.127, 0.185, -0.021), 'XYLD': (-0.07, 0.023, -0.093, 0.897, -0.077, 0.006), 'JEPI': (0.021, 0.054, -0.032, 0.64, 0.04, -0.019), 'PBP': (-0.06, 0.024, -0.084, 0.919, -0.077, 0.017), 'DIVO': (0.042, 0.028, 0.015, 0.429, 0.057, -0.015), 'SPYI': (0.026, 0.063, -0.037, 0.745, 0.053, -0.028), 'SCHD': (0.02, 0.011, 0.009, 0.447, 0.034, -0.013), 'VYM': (-0.008, 0.009, -0.016, 0.614, -0.009, 0.002), 'DVY': (-0.018, 0.011, -0.03, 0.619, -0.027, 0.009), 'SPHD': (-0.041, 0.014, -0.054, 0.644, -0.067, 0.027), 'NOBL': (-0.072, 0.008, -0.08, 0.783, -0.063, -0.009), 'RYLD': (-0.028, 0.036, -0.063, 0.732, -0.027, -0.0)}
M = (0, 0.127, -0.035, 0.014, 0.046)
print(f"{'fund':6}{'observed':>10}{'null(b)':>9}{'excess':>9}{'p':>7}"
      f"{'null(a,b)':>11}{'excess':>9}")
for f, (o, nb_, eb, p, nab, eab) in R_MATCH.items():
    print(f"{f:6}{o:+10.3f}{nb_:+9.3f}{eb:+9.3f}{p:7.3f}{nab:+11.3f}{eab:+9.3f}")
print()
print(f"funds reaching p < 0.05 vs a plain scaled index: {M[0]}/14 "
      f"(smallest p = {M[1]:.3f})")
print(f"median excess spread vs that twin             : {M[2]:+.3f}")
print(f"null(a,b) reproduces the observed spread to a median |error| of "
      f"{M[3]:.3f} (max {M[4]:.3f})")


fund    observed  null(b)   excess      p  null(a,b)   excess
QYLD      -0.062   +0.031   -0.093  0.886     -0.058   -0.004
JEPQ      +0.023   +0.052   -0.029  0.644     +0.069   -0.046
NUSI      +0.164   +0.047   +0.117  0.127     +0.185   -0.021
XYLD      -0.070   +0.023   -0.093  0.897     -0.077   +0.006
JEPI      +0.021   +0.054   -0.032  0.640     +0.040   -0.019
PBP       -0.060   +0.024   -0.084  0.919     -0.077   +0.017
DIVO      +0.042   +0.028   +0.015  0.429     +0.057   -0.015
SPYI      +0.026   +0.063   -0.037  0.745     +0.053   -0.028
SCHD      +0.020   +0.011   +0.009  0.447     +0.034   -0.013
VYM       -0.008   +0.009   -0.016  0.614     -0.009   +0.002
DVY       -0.018   +0.011   -0.030  0.619     -0.027   +0.009
SPHD      -0.041   +0.014   -0.054  0.644     -0.067   +0.027
NOBL      -0.072   +0.008   -0.080  0.783     -0.063   -0.009
RYLD      -0.028   +0.036   -0.063  0.732     -0.027   -0.000

funds reaching p < 0.05 vs a plain scaled index: 0/14 (smallest p = 0

Two readings, and the second is the sharper one:

1. **0 of 14** funds' spreads reach *p* < 0.05 against a plain scaled index (smallest 0.127); the median excess is **-0.035** — real, and about a third of what the unconditional bias would have implied.
2. The alpha-carrying twin reproduces every observed spread to a median error of **0.014** (max 0.046). The capture spread carries **no information beyond (alpha, beta)**. It is a re-encoding of the CAPM fit, not an independent measurement of payoff shape — which is the strongest form of this study's claim.

> 💡 **In plain words** — the industry's ruler reads long, and once you correct it against each fund's own history it turns out to be measuring the fund's alpha and beta all over again, in worse units.

## 3. Is it just beta? CAPM alpha, and the kink intercept

Single-beta market model on excess-of-cash monthly returns, HAC(6). The **kink intercept** is the piecewise fit's intercept — the fund's excess return in a flat benchmark month, i.e. the premium an option-writer collects. It is *not* alpha: a kinked fit lifts the intercept mechanically and the same fit hands it back through a truncated up-beta.

In [7]:
R_CAPM = {'QYLD': (0.52, -18.6, -1.24, 51.6), 'JEPQ': (0.65, 3.8, 0.22, 99.4), 'NUSI': (0.54, 33.6, 1.43, 81.0), 'XYLD': (0.68, -17.4, -1.17, 40.4), 'JEPI': (0.56, -2.6, -0.21, 23.3), 'PBP': (0.63, -18.8, -1.56, 48.8), 'DIVO': (0.73, 5.7, 0.45, 3.9), 'SPYI': (0.71, -1.6, -0.17, 58.9), 'SCHD': (0.83, 3.8, 0.23, 18.4), 'VYM': (0.87, -3.3, -0.28, 14.7), 'DVY': (0.83, -7.1, -0.34, 30.1), 'SPHD': (0.77, -13.9, -0.48, 31.1), 'NOBL': (0.84, -12.4, -0.81, 12.7), 'RYLD': (0.58, -16.5, -0.61, 120.1)}
print(f"{'fund':6}{'beta':>7}{'alpha bps/mo':>15}{'t':>8}{'kink bps/mo':>14}")
for f, (b, a, t, k) in R_CAPM.items():
    print(f"{f:6}{b:7.2f}{a:+15.1f}{t:+8.2f}{k:+14.1f}")
print(f"\nmax |t| on alpha across the panel: {R['max_abs_t_alpha']:.2f}"
      f"  -> no fund adds anything beyond a smaller index position")


fund     beta   alpha bps/mo       t   kink bps/mo
QYLD     0.52          -18.6   -1.24         +51.6
JEPQ     0.65           +3.8   +0.22         +99.4
NUSI     0.54          +33.6   +1.43         +81.0
XYLD     0.68          -17.4   -1.17         +40.4
JEPI     0.56           -2.6   -0.21         +23.3
PBP      0.63          -18.8   -1.56         +48.8
DIVO     0.73           +5.7   +0.45          +3.9
SPYI     0.71           -1.6   -0.17         +58.9
SCHD     0.83           +3.8   +0.23         +18.4
VYM      0.87           -3.3   -0.28         +14.7
DVY      0.83           -7.1   -0.34         +30.1
SPHD     0.77          -13.9   -0.48         +31.1
NOBL     0.84          -12.4   -0.81         +12.7
RYLD     0.58          -16.5   -0.61        +120.1

max |t| on alpha across the panel: 1.56  -> no fund adds anything beyond a smaller index position


## 4. Excess-of-cash Sharpe race (both sides excess of BIL)

`gap` = fund Sharpe − benchmark Sharpe; the *t* is HAC(6) on the **monthly return difference**, which is the Jobson-Korkie comparison in its Newey-West form (the cash leg cancels in the difference).

In [8]:
R_SH = {'QYLD': (0.633, 0.984, -0.351, -3.99), 'XYLD': (0.61, 0.896, -0.286, -3.94), 'PBP': (0.391, 0.678, -0.287, -3.66), 'SPYI': (0.976, 1.017, -0.041, -2.62), 'JEPQ': (0.977, 0.992, -0.015, -2.2), 'JEPI': (0.766, 0.941, -0.175, -2.13), 'NOBL': (0.639, 0.862, -0.223, -1.87), 'SPHD': (0.597, 0.944, -0.347, -1.6), 'RYLD': (0.259, 0.451, -0.193, -1.56), 'DIVO': (0.814, 0.847, -0.034, -1.46), 'VYM': (0.567, 0.644, -0.077, -1.22), 'DVY': (0.471, 0.644, -0.173, -1.16), 'SCHD': (0.858, 0.966, -0.109, -0.96), 'NUSI': (1.086, 0.92, 0.166, -1.28)}
print(f"{'fund':6}{'fund Sh':>10}{'bench Sh':>10}{'gap':>9}{'HAC t':>8}")
for f, (sf, sb, g, t) in R_SH.items():
    print(f"{f:6}{sf:+10.3f}{sb:+10.3f}{g:+9.3f}{t:+8.2f}")
print(f"\nfunds beating their own benchmark: "
      f"{R['n_beating_bench']}/{R['n_funds']}")


fund     fund Sh  bench Sh      gap   HAC t
QYLD      +0.633    +0.984   -0.351   -3.99
XYLD      +0.610    +0.896   -0.286   -3.94
PBP       +0.391    +0.678   -0.287   -3.66
SPYI      +0.976    +1.017   -0.041   -2.62
JEPQ      +0.977    +0.992   -0.015   -2.20
JEPI      +0.766    +0.941   -0.175   -2.13
NOBL      +0.639    +0.862   -0.223   -1.87
SPHD      +0.597    +0.944   -0.347   -1.60
RYLD      +0.259    +0.451   -0.193   -1.56
DIVO      +0.814    +0.847   -0.034   -1.46
VYM       +0.567    +0.644   -0.077   -1.22
DVY       +0.471    +0.644   -0.173   -1.16
SCHD      +0.858    +0.966   -0.109   -0.96
NUSI      +1.086    +0.920   +0.166   -1.28

funds beating their own benchmark: 1/14


## 5. Robustness — era cut, rank persistence, and the assumption sweeps

In [9]:
R_ERA = {'QYLD': (-0.107, -0.016), 'XYLD': (-0.193, 0.052), 'PBP': (-0.112, 0.073), 'DIVO': (0.017, 0.057), 'SCHD': (0.025, 0.002), 'VYM': (-0.037, 0.092), 'DVY': (-0.025, 0.112), 'SPHD': (-0.116, 0.034), 'NOBL': (-0.024, -0.125)}
RHO, PVAL, NOBS, AGREE = (-0.117, 0.756, 9, 0.44)
print('era cut, split 2021-01-01 (>= 24 months each side):')
for f, (a, b) in R_ERA.items():
    flag = 'SIGN FLIP' if a * b < 0 else ''
    print(f"  {f:6} early {a:+.3f}   late {b:+.3f}   {flag}")
print(f"\nSpearman rho = {RHO:+.3f} (p = {PVAL:.3f}, n = {NOBS}), "
      f"sign agreement {AGREE:.0%}")


era cut, split 2021-01-01 (>= 24 months each side):
  QYLD   early -0.107   late -0.016   
  XYLD   early -0.193   late +0.052   SIGN FLIP
  PBP    early -0.112   late +0.073   SIGN FLIP
  DIVO   early +0.017   late +0.057   
  SCHD   early +0.025   late +0.002   
  VYM    early -0.037   late +0.092   SIGN FLIP
  DVY    early -0.025   late +0.112   SIGN FLIP
  SPHD   early -0.116   late +0.034   SIGN FLIP
  NOBL   early -0.024   late -0.125   

Spearman rho = -0.117 (p = 0.756, n = 9), sign agreement 44%


In [10]:
SPY_MED, SPY_POS, SPY_NEG = (-0.003, 0, 5)
GEO = {'QYLD': (-0.062, -0.178), 'RYLD': (-0.028, -0.171), 'DIVO': (0.042, -0.038)}
print("ASSUMPTION 1 - the fund -> benchmark map (hand-assigned from each")
print("               fund's stated index). Sweep: force SPY for every fund.")
print(f"  median spread {SPY_MED:+.3f}  |  t_convexity >= +2: {SPY_POS}"
      f"  |  <= -2: {SPY_NEG}")
print()
print('ASSUMPTION 2 - the capture convention. Sweep: arithmetic vs geometric.')
print('  13 of 14 turn negative under the geometric convention')
for f, (a, g) in GEO.items():
    print(f"    {f:6} arithmetic {a:+.3f} -> geometric {g:+.3f}")


ASSUMPTION 1 - the fund -> benchmark map (hand-assigned from each
               fund's stated index). Sweep: force SPY for every fund.
  median spread -0.003  |  t_convexity >= +2: 0  |  <= -2: 5

ASSUMPTION 2 - the capture convention. Sweep: arithmetic vs geometric.
  13 of 14 turn negative under the geometric convention
    QYLD   arithmetic -0.062 -> geometric -0.178
    RYLD   arithmetic -0.028 -> geometric -0.171
    DIVO   arithmetic +0.042 -> geometric -0.038


Neither assumption moves the verdict: forcing SPY on everyone still leaves **zero** funds with a positive convexity *t*, and the geometric convention pushes **every** fund's spread down.

## 6. Tradability — the beta-neutral traded spread

Long the fund, short `beta_hat` of the benchmark, remainder in cash. `beta_hat` = 36-month rolling OLS slope **through month *t***, held over ***t*+1** (the one execution lag). Cost 5 bps one-way × NAV on the notional turned over; the short index leg pays borrow.

In [11]:
R_TR = {'QYLD': (114, -24.2, -1.36, -28.8, -1.62), 'XYLD': (120, -27.9, -1.87, -34.1, -2.28), 'PBP': (186, -18.9, -1.55, -24.0, -1.96), 'SCHD': (140, -4.1, -0.22, -11.6, -0.62), 'DVY': (193, 0.8, 0.04, -6.0, -0.29)}
BEST, BEST_BPS, BEST_T = ('DVY', 0.8, 0.04)
BORROW = [(0, 0.6, 0.03, 0.01), (50, -2.7, -0.13, -0.04), (100, -6.0, -0.29, -0.08), (200, -12.7, -0.62, -0.18)]
print(f"{'fund':6}{'n':>5}{'gross bps/mo':>15}{'t':>8}"
      f"{'net(5bps,100bp)':>18}{'t':>8}")
for f, (n, g, tg, nt, tn) in R_TR.items():
    print(f"{f:6}{n:5d}{g:+15.1f}{tg:+8.2f}{nt:+18.1f}{tn:+8.2f}")
ANY = ('NUSI', 53.0, 1.6, 42)
print(f"\nlargest gross spread in the panel, no history floor: {ANY[0]} at "
      f"{ANY[1]:+.1f} bps/mo on {ANY[3]} months (t = {ANY[2]:+.2f})")
print(f"best spread in the panel with >= 60 months: {BEST} at "
      f"{BEST_BPS:+.1f} bps/mo gross (t = {BEST_T:+.2f})")
print('\nborrow sweep on that best case:')
for b, m, t, s in BORROW:
    print(f"  borrow {b:4d} bps -> {m:+6.1f} bps/mo "
          f"(t = {t:+.2f}, Sharpe {s:+.2f})")


fund      n   gross bps/mo       t   net(5bps,100bp)       t
QYLD    114          -24.2   -1.36             -28.8   -1.62
XYLD    120          -27.9   -1.87             -34.1   -2.28
PBP     186          -18.9   -1.55             -24.0   -1.96
SCHD    140           -4.1   -0.22             -11.6   -0.62
DVY     193           +0.8   +0.04              -6.0   -0.29

largest gross spread in the panel, no history floor: NUSI at +53.0 bps/mo on 42 months (t = +1.60)
best spread in the panel with >= 60 months: DVY at +0.8 bps/mo gross (t = +0.04)

borrow sweep on that best case:
  borrow    0 bps ->   +0.6 bps/mo (t = +0.03, Sharpe +0.01)
  borrow   50 bps ->   -2.7 bps/mo (t = -0.13, Sharpe -0.04)
  borrow  100 bps ->   -6.0 bps/mo (t = -0.29, Sharpe -0.08)
  borrow  200 bps ->  -12.7 bps/mo (t = -0.62, Sharpe -0.18)


> 💡 **In plain words** — the best beta-neutral version of this trade with a real history makes less than a basis point a month before the short leg is paid for, and loses money once it is. The one that looks better (NUSI, 42 months) is short, not significant, and belongs to the fund whose tape needed a split repaired.

## 7. Live synthetic control — the machinery is unbiased

**Synthetic by construction.** A planted cross-section (one genuinely convex fund, three genuinely concave, three flat) must be recovered; a null cross-section (every fund plain linear beta) must stay quiet at roughly the nominal rate. This proves the real-tape zero is a property of the funds, not of the harness — it never supports the stamp.

In [12]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from capture_ratio import data, strategy as st
planted, tp = data.synthetic_panel(signal_strength=1.0, seed=948)
dp = st.synthetic_detect(planted, tp, n_boot=200)
print(f"planted: corr(measured spread, planted spread) = {dp['corr_measured_true']:+.3f}")
print(f"         {dp['n_hits_positive']} positive / {dp['n_hits_negative']} negative "
      f"convexity hits of {dp['n_funds']} funds (planted: 1 convex, 3 concave)")
tbl = dp['table'][['true_spread', 'spread', 'convexity', 't_convexity']]
print(tbl.round(3).to_string())
fires = 0
for s in range(6):
    z, tz = data.synthetic_panel(signal_strength=0.0, seed=948 + s)
    d = st.synthetic_detect(z, tz, n_boot=1)
    fires += d['n_hits_positive'] + d['n_hits_negative']
print(f"\nnull x6 seeds: |t| >= 2 fires on {fires}/42 fund-tests "
      f"(nominal 5% predicts ~2.1)")


planted: corr(measured spread, planted spread) = +0.912
         1 positive / 3 negative convexity hits of 7 funds (planted: 1 convex, 3 concave)
           true_spread  spread  convexity  t_convexity
fund                                                  
CONVEX             0.3   0.409      0.255        4.009
FLAT_A             0.0   0.067      0.045        0.693
FLAT_B             0.0   0.034     -0.042       -0.653
FLAT_C             0.0  -0.061     -0.071       -1.239
CONCAVE_A         -0.3  -0.070     -0.269       -4.344
CONCAVE_B         -0.2  -0.081     -0.249       -4.092
CONCAVE_C         -0.4  -0.210     -0.310       -5.674



null x6 seeds: |t| >= 2 fires on 2/42 fund-tests (nominal 5% predicts ~2.1)


## Verdict

- **Signal — None.** The estimand is the capture spread, and it is zero. **0 of 14** funds reach a nominal positive |*t*| ≥ 2 on the convexity coefficient; **0** clear the family-wise bar of 2.91; the bootstrap CI covers zero for 14/14; against each fund's own zero-convexity twin, on its own sample and the real benchmark path, 0/14 reach *p* < 0.05 (smallest 0.127) and the median excess spread is -0.035; and the ranking has no era-to-era persistence (rho = -0.117, *p* = 0.76). The largest positive (NUSI, +0.164) needed an unadjusted reverse split removed before it stopped reading +0.939, and still carries a *negative* convexity coefficient. What is robust is the opposite sign: 6 significantly concave funds, all option-writers — an accounting identity of selling calls. **Survivorship** is named here: the panel is the funds still listed at the as-of, so the closed ones flatter it.
- **Tradability — Mirage.** Beta 0.52–0.87, no alpha anywhere (max |*t*| = 1.56, and negative), 1/14 beating its benchmark on excess-of-cash Sharpe — NUSI, on lower vol, with a HAC *t* of −1.28 on the monthly return difference. The beta-neutral arm tops out at +53.0 bps/month gross on 42 months (*t* = +1.60) and, among funds with a real history, at +0.8 bps/month (*t* = +0.04) — which dies at 50 bps of borrow. The drawdown reduction is real and is bought with beta, which is available for 3 bps.
- **Methodological note worth keeping.** The arithmetic capture spread is a **biased** estimator — positive 82% of the time on a truth of exactly zero — and once that bias is measured against each fund's own sample it turns out the spread reproduces the fund's (alpha, beta) fit to within 0.014. Any fund scorecard quoting up- and down-capture without an interval is reporting a CAPM regression in worse units.